In [1]:
import os
import torch
import clip 
from PIL import Image
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import pickle
from torchvision import transforms

In [2]:
DATA_DIR = "data/raw_images"
ARTIFACTS_DIR = "artifacts"
REF_DATA_FILE = "data/ref_data.csv"
DEVICE = "cpu"

os.makedirs(ARTIFACTS_DIR, exist_ok=True)

In [3]:
# EXTRACTION DES FEATURES
def get_features(root_dir):
    features = []
    labels = []
    classes = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
    
    print(f"Traitement des {len(classes)} classes avec CLIP")
    
    for label_str in classes:
        class_dir = os.path.join(root_dir, label_str)
        images = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        for img_name in images:
            img_path = os.path.join(class_dir, img_name)
            try:
                img = Image.open(img_path).convert('RGB')
                img_preprocessed = preprocess(img).unsqueeze(0).to(DEVICE)
                
                for _ in range(3): 
                    img_aug = augmenter(img)
                    
                    img_input = preprocess(img_aug).unsqueeze(0).to(DEVICE)
                    with torch.no_grad():
                        vector = model.encode_image(img_input).cpu().numpy().flatten()
                    
                    features.append(vector)
                    labels.append(label_str)
            except Exception as e:
                # print(f"Erreur sur {img_name}")
                pass
                
    return np.array(features), np.array(labels)


In [4]:
print("Chargement de CLIP (ViT-B/32)")

# model est le réseau de neurones, preprocess est la transformation d'image obligatoire
model, preprocess = clip.load("ViT-B/32", device=DEVICE)
augmenter = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),     
    transforms.RandomRotation(degrees=15),      
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomGrayscale(p=0.1),           
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0))
])

Chargement de CLIP (ViT-B/32)


In [5]:
for f in os.listdir(ARTIFACTS_DIR):
    os.remove(os.path.join(ARTIFACTS_DIR, f))

In [6]:
X_raw, y = get_features(DATA_DIR)
print(f"Données extraites : {X_raw.shape} (512 dimensions par image)")


Traitement des 81 classes avec CLIP
Données extraites : (48600, 512) (512 dimensions par image)


In [7]:
print("Normalisation et Réduction PCA...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)


pca = PCA(n_components=0.95) # 95% de la variance expliquée
X_pca = pca.fit_transform(X_scaled)
print(f"   Dimensions avant: {X_scaled.shape} -> Après PCA: {X_pca.shape}")

cols = [f"PCA_{i}" for i in range(X_pca.shape[1])]
df_ref = pd.DataFrame(X_pca, columns=cols)
df_ref['target'] = y
df_ref.to_csv(REF_DATA_FILE, index=False)


Normalisation et Réduction PCA...
   Dimensions avant: (48600, 512) -> Après PCA: (48600, 330)


In [8]:
print("Entraînement XGBoost sur les données réduites")
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X_pca, y_encoded, test_size=0.2, random_state=42)

clf = XGBClassifier(n_estimators=200, max_depth=5, use_label_encoder=False, eval_metric='mlogloss')
clf.fit(X_train, y_train)


Entraînement XGBoost sur les données réduites


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [23:21:29] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [9]:
print(classification_report(label_encoder.inverse_transform(y_test), 
                            label_encoder.inverse_transform(clf.predict(X_test))))

with open(os.path.join(ARTIFACTS_DIR, "model.pickle"), "wb") as f: pickle.dump(clf, f)
with open(os.path.join(ARTIFACTS_DIR, "scaler.pickle"), "wb") as f: pickle.dump(scaler, f)
with open(os.path.join(ARTIFACTS_DIR, "pca.pickle"), "wb") as f: pickle.dump(pca, f)
with open(os.path.join(ARTIFACTS_DIR, "label_encoder.pickle"), "wb") as f: pickle.dump(label_encoder, f)

print("✨ Modèle CLIP Ready terminé.")

                                       precision    recall  f1-score   support

        Am_I_The_Only_One_Around_Here       0.97      0.97      0.97       112
            American_Chopper_Argument       0.99      0.96      0.98       125
                       Ancient_Aliens       0.92      0.98      0.95       101
                       Bad_Luck_Brian       0.97      0.96      0.96       113
                Batman_Slapping_Robin       0.96      0.96      0.96       108
                         Be_Like_Bill       0.98      0.98      0.98       126
                       Black_Girl_Wat       0.95      0.99      0.97       113
                     Blank_Nut_Button       0.95      0.93      0.94       131
         Boardroom_Meeting_Suggestion       0.98      0.98      0.98       118
         Brace_Yourselves_X_is_Coming       0.97      0.96      0.97       112
              Captain_Picard_Facepalm       0.96      0.96      0.96       120
                       Change_My_Mind       0.99   